## Qwen2-VL Prompt v2: Train Validation 50개 재평가

이 Notebook은 대회 제공 `Qwen2-VL-2B-Instruct`의 모델 가중치는 그대로 유지하고, identity 편향을 제거한 Prompt v2를 고정된 **Train Validation**에서 평가합니다. Test 데이터는 읽지 않습니다. 기존 Prompt v1 결과는 덮어쓰지 않습니다.

진행 순서:

1. Colab GPU 확인
2. 비공개 Google Drive의 프로젝트 연결
3. Qwen 실행 패키지 설치
4. parser 관련 테스트 실행
5. seed 42로 선택한 Validation 50개 평가
6. 결과가 안정적일 때만 전체 Validation 1,907개 평가

대회 데이터가 있는 Drive 폴더를 공개하거나 공유하지 마세요.

### 1. GPU 확인

Colab 메뉴에서 `런타임 → 런타임 유형 변경 → GPU`를 선택한 뒤 실행합니다.

In [ ]:
import torch

assert torch.cuda.is_available(), 'GPU runtime이 아닙니다. Colab 런타임 유형을 GPU로 변경하세요.'
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

### 2. 프로젝트 경로 연결

`PROJECT_DIR`을 실제 비공개 Drive 경로에 맞게 수정하세요. 해당 폴더 안에는 현재 프로젝트 코드, `data/train.csv`, `data/train/`, `outputs/splits/validation_ids.csv`가 있어야 합니다.

In [ ]:
import os
import sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# 본인의 Google Drive 경로에 맞게 이 한 줄만 수정하세요.
PROJECT_DIR = Path('/content/drive/MyDrive/SNU')

assert (PROJECT_DIR / 'data' / 'train.csv').is_file(), 'data/train.csv를 찾지 못했습니다.'
assert (PROJECT_DIR / 'data' / 'train').is_dir(), 'data/train 폴더를 찾지 못했습니다.'
assert (PROJECT_DIR / 'outputs' / 'splits' / 'validation_ids.csv').is_file(), 'Validation 분할 파일을 찾지 못했습니다.'
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print(f'Project: {PROJECT_DIR}')

### 3. Qwen 실행 패키지 설치

Colab에 이미 설치된 CUDA용 PyTorch는 교체하지 않고 Qwen 실행에 필요한 패키지만 설치합니다.

In [ ]:
%pip install -q transformers==5.13.0 accelerate==1.14.0 qwen-vl-utils==0.0.14

### 4. Qwen baseline 변환 테스트

모델을 받기 전에 24개 label 변환, parser, prompt 순서, 평가 요약 테스트를 실행합니다. 오류가 나면 이 셀이 실제 테스트 원인까지 출력합니다.

In [ ]:
import subprocess
import sys

test_file = PROJECT_DIR / "tests" / "test_qwen_baseline.py"
assert test_file.is_file(), f"테스트 파일을 찾지 못했습니다: {test_file}"

subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        str(PROJECT_DIR / "tests"),
        "-p",
        "test_qwen_baseline.py",
        "-v",
    ],
    cwd=PROJECT_DIR,
    check=True,
)

### 5. Prompt v2로 동일한 Validation 50개 zero-shot 평가

첫 실행에서는 약 4.4GB의 Qwen 모델을 `/content`에 다운로드합니다. Prompt v1과 동일한 seed 42의 50개 Validation 샘플을 사용해 prompt 변경 효과만 비교합니다. Prompt v2는 구체적인 identity 예시를 제거하고 대회 `Answer`를 직접 출력합니다.

In [ ]:
CACHE_DIR = Path('/content/huggingface/hub')
OUTPUT_BASE = PROJECT_DIR / 'outputs' / 'qwen_zero_shot_prompt_v2'

sample_command = [
    sys.executable, '-m', 'scripts.evaluate_qwen_baseline',
    '--data-dir', str(PROJECT_DIR / 'data'),
    '--validation-split-csv', str(PROJECT_DIR / 'outputs' / 'splits' / 'validation_ids.csv'),
    '--output-dir', str(OUTPUT_BASE),
    '--cache-dir', str(CACHE_DIR),
    '--device', 'cuda',
    '--prompt-version', 'v2_direct_answer',
    '--limit', '50',
    '--seed', '42',
    '--max-new-tokens', '16',
]
subprocess.run(sample_command, check=True)

### 6. 50개 결과 확인

Prompt v2의 점수, `[1,2,3,4]` baseline, parsing 성공률, 예측 쏠림과 latency를 확인합니다. 기존 Prompt v1 결과가 Drive에 있으면 핵심 지표를 나란히 비교합니다.

In [ ]:
import json
import pandas as pd
from IPython.display import display

sample_summary_path = OUTPUT_BASE / 'validation_50' / 'summary.json'
sample_predictions_path = OUTPUT_BASE / 'validation_50' / 'predictions.csv'
sample_summary = json.loads(sample_summary_path.read_text(encoding='utf-8'))

print(json.dumps(sample_summary, ensure_ascii=False, indent=2))
display(pd.read_csv(sample_predictions_path).head(10))

v1_summary_path = PROJECT_DIR / 'outputs' / 'qwen_zero_shot' / 'validation_50' / 'summary.json'
if v1_summary_path.is_file():
    v1_summary = json.loads(v1_summary_path.read_text(encoding='utf-8'))
    comparison = pd.DataFrame([
        {
            'prompt': 'v1_chronological',
            'exact_match': v1_summary['metrics']['exact_match_accuracy'],
            'parse_success': v1_summary['parsing']['success_rate'],
            'unique_answers': v1_summary['predictions']['unique_answer_count'],
            'most_common_fraction': v1_summary['predictions']['most_common_fraction'],
        },
        {
            'prompt': 'v2_direct_answer',
            'exact_match': sample_summary['metrics']['exact_match_accuracy'],
            'parse_success': sample_summary['parsing']['success_rate'],
            'unique_answers': sample_summary['predictions']['unique_answer_count'],
            'most_common_fraction': sample_summary['predictions']['most_common_fraction'],
        },
    ])
    display(comparison)
else:
    print('기존 Prompt v1 summary가 없어 v2 결과만 표시했습니다.')

### 7. 전체 Validation 1,907개 평가 — 결과 확인 후에만 실행

50개 실행에서 parsing 성공률과 속도가 괜찮은 것을 확인한 뒤 `RUN_FULL_VALIDATION = True`로 변경하세요. 같은 Colab runtime에서는 이미 받은 모델 cache를 offline으로 재사용합니다.

In [ ]:
RUN_FULL_VALIDATION = False

if RUN_FULL_VALIDATION:
    full_command = [
        sys.executable, '-m', 'scripts.evaluate_qwen_baseline',
        '--data-dir', str(PROJECT_DIR / 'data'),
        '--validation-split-csv', str(PROJECT_DIR / 'outputs' / 'splits' / 'validation_ids.csv'),
        '--output-dir', str(OUTPUT_BASE),
        '--cache-dir', str(CACHE_DIR),
        '--device', 'cuda',
        '--prompt-version', 'v2_direct_answer',
        '--limit', '0',
        '--seed', '42',
        '--max-new-tokens', '16',
        '--local-files-only',
    ]
    subprocess.run(full_command, check=True)
else:
    print('50개 결과를 먼저 확인하세요. 전체 Validation은 실행하지 않았습니다.')

### 결과 판단 기준

- `parsing.success_rate`가 낮으면 성능보다 prompt 또는 parser를 먼저 수정해야 합니다.
- `predictions.unique_answer_count`가 1에 가까우면 예측 collapse를 의심합니다.
- 50개 표본의 Exact Match는 변동성이 크므로 모델 선택에 사용하지 않습니다.
- 최종 비교에는 전체 Validation 1,907개의 `metrics.exact_match_accuracy`를 사용합니다.
- 현재 전체 Validation identity baseline은 약 15.47%입니다.
- Qwen과 CLIP의 예측을 합치지 않으며, Validation 결과가 더 좋은 단일 모델 방향만 선택합니다.